# IS2083 Lab 5: AI Automation Tool, Part 1

**Before you begin, you need TWO things:**

1. Your **Jupyter login** (you used it to open this notebook).
2. Your **API key** for the class Qwen Coder model server (from your instructor).

This lab assumes you already know basic Python. The new material is the **AI model**, its **API**, and **prompt design**. You will connect to the class **Qwen Coder** model, use it as a coding assistant, and build an **AI-powered security log assessment** tool. At the end you will feed it a real security log exported from **Windows** or **macOS** and have it write an assessment report. Run cells with **Shift + Enter**, top to bottom.

## Step 1: Install the openai client (run once)

In [1]:
%pip install openai

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## Step 2: Connect to the Qwen Coder model server

Paste the `BASE_URL` (ends in `/v1`) and your `API_KEY` from your instructor. The code then queries the server for the served model id.

In [ ]:
from openai import OpenAI

BASE_URL = "http://gpustack.cds.utsa.edu/v1"   # class model server address
API_KEY  = "gpustack_REDACTED"  # your personal API key

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
MODEL = client.models.list().data[1].id   # discover the served model id
print("Connected. Model:", MODEL)

Connected. Model: qwen3.5-122b-a10b


## Step 3: A reusable model helper (with retries)

`ask_model` wraps one chat completion and retries on transient errors.

In [3]:
import time

def ask_model(prompt: str, temperature: float = 1.0, retries: int = 2) -> str:
    """Send one prompt to Qwen Coder and return its text reply, retrying on failure."""
    for attempt in range(retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                top_p=0.95,
            )
            return resp.choices[0].message.content
        except Exception:
            if attempt == retries:
                raise
            time.sleep(1.5 * (attempt + 1))

print(ask_model("Say hello in one short sentence."))



Hello!


## Step 4: Plan the tool

Input: one security log entry. Output: a structured result with a `label` (BENIGN / SUSPICIOUS / MALICIOUS) and a one-sentence `reason`. You solve the structure problem with a strict output format in the prompt plus defensive parsing.

## Step 5: Use Qwen Coder as a coding assistant

Ask the model to sketch an approach, then keep only what you understand.

In [5]:
question = (
    "I am writing a Python function that asks an LLM to classify a security log entry "
    "and returns a dataclass with label and reason. Suggest a robust prompt format and "
    "a defensive way to parse the reply. Keep it concise."
)
print(ask_model(question))



```python
from dataclasses import dataclass
import json
import re

@dataclass
class LogAnalysis:
    label: str
    reason: str

PROMPT = """
You are a security log analyst. Classify the following log entry.
Output format: JSON with keys 'label' (e.g., 'Malicious', 'Suspicious', 'Benign') and 'reason'.
Respond with VALID JSON ONLY. No markdown.

Log: {log_entry}"""

def parse_llm_response(raw_text: str) -> LogAnalysis:
    try:
        # 1. Strip markdown code blocks if present
        clean = re.sub(r"```json\s*|\s*```", "", raw_text.strip())
        # 2. Parse JSON
        data = json.loads(clean)
        # 3. Validate schema
        if "label" not in data or "reason" not in data:
            raise ValueError("Missing required fields")
        return LogAnalysis(label=data["label"], reason=data["reason"])
    except Exception as e:
        # Fallback or re-raise with context
        raise RuntimeError(f"Failed to parse LLM response: {e}")

# Usage
def classify_log(client, log_entry

## Step 6: Define the result type and parsers

An `Enum` fixes the labels, a `dataclass` gives a typed result, and two regexes extract the fields.

In [6]:
from dataclasses import dataclass
from enum import Enum
import re

class Label(str, Enum):
    BENIGN = "BENIGN"
    SUSPICIOUS = "SUSPICIOUS"
    MALICIOUS = "MALICIOUS"
    UNKNOWN = "UNKNOWN"

@dataclass
class Triage:
    line: str
    label: Label
    reason: str

LABEL_RE = re.compile(r"^LABEL:\s*(\w+)", re.IGNORECASE | re.MULTILINE)
REASON_RE = re.compile(r"^REASON:\s*(.+)", re.IGNORECASE | re.MULTILINE)

## Step 7: Name your tool

This tool can go in your professional portfolio, so give it a name you would be proud to show. `triage_log` is a fine name for the function, but the **tool** itself can be called anything: Play it straight (SentinelScan, ThreatLens, LogHawk, WatchTower) or give it some personality (SnitchBot, LogFather, GrepReaper, Sherlock Logs, The Log Whisperer, LogNessMonster), your call. Set it here; it appears in your report and your repository name.

In [11]:
TOOL_NAME = "RoostSweep"   # <-- change this to your own creative name

## Step 8: Build the tool

Write the classifier: a strict prompt, one `ask_model` call, then defensive parsing that falls back to `Label.UNKNOWN`. (You may rename `triage_log` to match your tool if you like.)

In [7]:
def triage_log(line: str) -> Triage:
    prompt = (
        "You are a security log triage assistant.\n"
        "Classify the log line as BENIGN, SUSPICIOUS, or MALICIOUS and give a one "
        "sentence reason.\n"
        "Reply in EXACTLY this format:\n"
        "LABEL: <BENIGN|SUSPICIOUS|MALICIOUS>\n"
        "REASON: <one sentence>\n\n"
        f"Log entry:\n{line}\n"
    )
    text = ask_model(prompt)

    label_match = LABEL_RE.search(text)
    raw = label_match.group(1).upper() if label_match else "UNKNOWN"
    label = Label(raw) if raw in Label.__members__ else Label.UNKNOWN

    reason_match = REASON_RE.search(text)
    reason = reason_match.group(1).strip() if reason_match else text.strip()

    return Triage(line=line, label=label, reason=reason)

## Step 9: Quick sanity check
Confirm the tool works on one entry before feeding it a file.

In [8]:
print(triage_log("Failed password for root from 203.0.113.9 port 4022 ssh2"))

Triage(line='Failed password for root from 203.0.113.9 port 4022 ssh2', label=<Label.SUSPICIOUS: 'SUSPICIOUS'>, reason='This log indicates a failed authentication attempt against the privileged root account, which is a common indicator of SSH brute-force attacks.')


## Step 10: Assess a real security log file

Export your computer's security log to **one file** and upload it here. Do the ONE that matches your computer; you only turn in one (Windows or macOS).

**Windows (Event Viewer):** open Event Viewer, expand **Windows Logs**, click **Security**. In the right panel click **Save All Events As...**, set *Save as type* to **CSV (Comma Separated)**, and save it as `security_log.csv`.

**macOS (Terminal):** export recent security-related events to one file:

```
log show --predicate 'process == "sudo" OR process == "sshd" OR process == "loginwindow" OR eventMessage CONTAINS[c] "authentication"' --last 6h --style syslog > security_log.txt
```

If that file is nearly empty, use: `log show --last 1h --style syslog > security_log.txt`

Then click **Upload** in the Jupyter file browser and add your `security_log.csv` (Windows) or `security_log.txt` (macOS) to this notebook's folder. Set `LOG_FILE` below to match.

In [9]:
LOG_FILE = "security_log.csv"   # Windows CSV, or "security_log.txt" on macOS
MAX_ENTRIES = 20                 # keep this first version small and quick

import csv

def read_log_lines(path, limit=MAX_ENTRIES):
    with open(path, encoding="utf-8", errors="ignore", newline="") as f:
        rows = list(csv.reader(f))[1:] # skip the header row
    events = [" | ".join(" ".join(c.split()) for c in row) for row in rows if row]
    return events[:limit]

entries = read_log_lines(LOG_FILE)
results = [triage_log(line) for line in entries]
print(f"Assessed {len(results)} entries from {LOG_FILE}")

Assessed 20 entries from security_log.csv


## Step 11: Write the assessment report

Have your tool write a Markdown report named after your tool. This report is your deliverable.

In [12]:
from datetime import datetime

def write_report(results, tool_name, path="assessment.md"):
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"# {tool_name} Security Log Assessment\n\n")
        f.write(f"Generated: {datetime.now():%Y-%m-%d %H:%M}\n\n")
        f.write(f"Entries assessed: {len(results)}\n\n")
        for r in results:
            f.write(f"- **{r.label.value}** {r.reason}\n")
            f.write(f"  - `{r.line}`\n")
    return path

report_path = write_report(results, TOOL_NAME)
print("Wrote", report_path, "\n")
print(open(report_path, encoding="utf-8").read())

Wrote assessment.md 

# RoostSweep Security Log Assessment

Generated: 2026-11-10 13:32

Entries assessed: 20

- **BENIGN** The event is triggered by a legitimate Git Bash executable in a standard installation directory, suggesting normal development or scripting activity.
  - `Audit Success | 11/10/2026 1:14:05 PM | Microsoft-Windows-Security-Auditing | 4798 | User Account Management | A user's local group membership was enumerated. Subject: Security ID: BRUH\rylan Account Name: rylan Account Domain: BRUH Logon ID: 0x3ECDF User: Security ID: BRUH\rylan Account Name: rylan Account Domain: Bruh Process Information: Process ID: 0x3e18 Process Name: C:\Program Files\Git\usr\bin\bash.exe`
- **BENIGN** The activity originates from a legitimate Git Bash executable, which commonly queries group membership during standard development environment script execution.
  - `Audit Success | 11/10/2026 1:13:35 PM | Microsoft-Windows-Security-Auditing | 4798 | User Account Management | A user's local g

## Step 12: Read, test, and trust

Double-click to edit and answer briefly:

1. Why model the output with an `Enum` and `dataclass` instead of a bare dict?
2. What does your defensive parsing do when the model ignores the format?
3. Which entries in your report look wrong, and how would you catch that?

_Your answers:_
1. The Enum limits labels to four fixed values, so typos or made-up categories
   can't get in, and the dataclass gives every result the same three named
   fields (line, label, reason) instead of a loose dict.
   
2. It makes it so that if the model ignores the format, that one entry is
   replaced with UNKNOWN and the other entries still get processed.

3. All entries in the report look correct in formatting. The AI did however
   label entry 18 as SUSPICIOUS even though it's the same as the other
   entries before and after it. I'd catch it by rerunning any entry that
   gets flagged as anything but BENIGN.

## Step 13: Save and submit

1. **File > Save Notebook**, then **File > Download** for the `.ipynb`.
2. Download your report `assessment.md` too.
3. On GitHub, create a **public** repository **named after your tool** (for example `sentinelscan`). Do **not** name it lab5 or use the course name.
4. **Add file > Upload files**: upload your notebook and `assessment.md`. Do **not** upload the raw security log; it may contain private data.
5. Before uploading, **remove your real API key** from the connect cell.
6. Take the screenshots listed in the handout into the submission template.